# Replication gate part 2: re-discover the P heads by patching (Qwen2.5-VL-3B)

D40 part 2. `replication_flip_rate.ipynb` ablated Lietzow's **published** heads.
This notebook finds the heads **without being told them**: it runs their
discovery procedure over all 36 x 16 = 576 heads and compares what comes out
with their Table 5 (`configs/heads_lietzow_qwen3b.json`).

The method is a port of their repo (MIT, commit 145b453), in
`src/circuit_overlap/patching.py`:

1. For each correctly-conflicting object: the **Prior** prompt (source) and the
   **Visual** prompt (target), both on the counterfactual image.
2. For every head, paste the source run's head output (o_proj input slice, last
   token) into the target run (**P2V**), and the target's into the source
   (**V2P**). Record the logits of the two runs' answers.
3. Restoration score per head and direction, averaged over objects.
4. PCA on the 576 (P2V, V2P) points; heads beyond +2 sigma on PC1 are
   **promoting**, below -2 sigma **suppressing**.

Lietzow run a full forward pass per patched head (~84,000 passes here, ~70 h
on a T4). A last-token patch cannot change earlier positions, so this notebook
runs each prompt once without its last token, caches that, and re-runs only
the last token per patch. Section 2 checks that this equals a full pass.

**Pre-registered (DECISIONS.md D40, 2026-10-27, before this ran):** (c) >= 5 of
their 9 promoting heads in our promoting set, (d) Jaccard beats chance at
p < 0.001, (e) median PC1 rank of their 9 <= 29. Do not change them after
seeing the output; `scripts/analyze_replication_gate.py` applies them.

**Runtime:** about 45 min for the 469 objects' two prompt runs, plus the sweep
over the ~73 conflicting objects -- section 2 measures one patch and prints the
estimate (expect roughly 1.5-3 h). Run it as **Save Version -> Save & Run All**.
Results stream to `/kaggle/working/patching/*.jsonl`; delete that folder if the
prompts or the model change.

Settings: GPU T4, Internet ON.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file and new names would appear
# to be missing.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
# Otherwise a missing field surfaces much later as a confusing TypeError.
from dataclasses import fields as _fields
from circuit_overlap.notes import NoteSpec as _NoteSpec
from circuit_overlap.t_screen import ScreenConfig as _Cfg
import inspect as _inspect
from circuit_overlap.t_screen import run_screen as _run

_need = {
    "NoteSpec.mode / .uppercase": {"mode", "uppercase"} - {f.name for f in _fields(_NoteSpec)},
    "ScreenConfig.variants / .batch_size": {"variants", "batch_size"} - {f.name for f in _fields(_Cfg)},
    "run_screen(checkpoint_path=...)": {"checkpoint_path"} - set(_inspect.signature(_run).parameters),
}
import importlib.util as _ilu
for _mod in ("circuit_overlap.patching", "circuit_overlap.replication"):
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = {"module"}
_missing = {k: sorted(v) for k, v in _need.items() if v}
if _missing:
    raise RuntimeError(
        f"checkout {COMMIT} is missing {_missing}. "
        "Push the local commits to origin/main, then re-run this cell."
    )
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). transformers may cache
# "is bitsandbytes available?" at import time, so if the 4-bit load still
# fails right after installing, restart the kernel and run from cell 1.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
from circuit_overlap.t_screen import native_bf16
# Not torch.cuda.is_bf16_supported(): it says True on a T4 because it counts
# slow emulation (CLAUDE.md gotchas).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

# Visual-Counterfact is read with an HF token (Lietzow's README).
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
from circuit_overlap.heads import load_head_set, format_head_label
hs = load_head_set(f"{REPO_DIR}/configs/heads_lietzow_qwen3b.json")
print(hs.name, hs.checkpoint)
print("promoting :", [format_head_label(h) for h in hs.promoting])
print("suppressing:", [format_head_label(h) for h in hs.suppressing])
print("layer range:", hs.layer_range(), " reported prior flip: 84.9%")

In [ ]:
from circuit_overlap.t_screen import ScreenConfig, VISUAL_PROMPT, PRIOR_PROMPT, load_model
config = ScreenConfig()
model, processor = load_model(config)        # 4-bit, nf4, double quant (D10)
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

# [VERIFY] the head geometry before ablating anything: a mismatch here is the
# single most likely cause of a failed gate.
text_config = model.model.language_model.config
n_layers = text_config.num_hidden_layers
n_heads = text_config.num_attention_heads
head_dim = getattr(text_config, "head_dim", text_config.hidden_size // n_heads)
print(f"{n_layers} layers x {n_heads} heads, head_dim={head_dim}")
load_head_set(f"{REPO_DIR}/configs/heads_lietzow_qwen3b.json",
              n_layers=n_layers, n_heads_per_layer=n_heads)   # raises if 1-based

## 1b. [VERIFY] Image size and speed

Two things made the first run ~30x too slow, and both are cheap to rule out
here:

* **Image token budget.** `load_model` pins the image area with
  `min_pixels == max_pixels` (~1,280 image tokens, as Lietzow do). Newer
  transformers warns that those arguments are deprecated; if they are ignored,
  images go in at full resolution with many times more tokens. The cell
  rebuilds the processor with the newer `size=` argument if so.
* **Device.** The model must be on the GPU.

Do not run the sweep until the forward pass is a few seconds at most.

In [ ]:
import time, torch
from datasets import load_dataset
from transformers import AutoProcessor

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_probe = ds[0]["counterfact_image"].convert("RGB")

def build_inputs(image, prompt):
    messages = [{"role": "user",
                 "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[image], return_tensors="pt").to(model.device)

def n_image_tokens(image):
    t, h, w = build_inputs(image, "x")["image_grid_thw"][0].tolist()
    return t * h * w // 4                    # 2x2 patch merge -> LLM tokens

EXPECTED = config.pixels // (28 * 28)        # 1280
n_img = n_image_tokens(_probe)
print(f"image {_probe.size} -> {n_img} image tokens (expected ~{EXPECTED})")
if n_img > 1.2 * EXPECTED:
    # min_pixels/max_pixels were ignored; the newer API takes them as `size`.
    processor = AutoProcessor.from_pretrained(
        config.checkpoint, use_fast=True,
        size={"shortest_edge": config.pixels, "longest_edge": config.pixels},
    )
    n_img = n_image_tokens(_probe)
    print(f"rebuilt the processor with size=... -> {n_img} image tokens")
assert n_img <= 1.2 * EXPECTED, "image token budget is still wrong -- do not run the loops"

assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 and restart"
_inp = build_inputs(_probe, VISUAL_PROMPT.format(object="banana"))
for attempt in ("warm-up", "timed"):
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        model(**_inp, use_cache=False)
    torch.cuda.synchronize()
    print(f"{attempt} forward pass: {time.time() - t0:.1f}s")
print("expect the timed pass to take ~0.5-3 s on a T4")

## 2. Objects, and a check of the cached path

Same row filters and colour tokens as `replication_flip_rate.ipynb`
(`replication.lietzow_rows`, `color_first_token_ids`). The cached last-token
path must match a full forward pass, both plain and with one head patched;
the cell stops the notebook if it does not.

In [ ]:
import inflect, pathlib, time
import torch
from circuit_overlap.replication import lietzow_rows, is_correctly_conflicting, run_cached
from circuit_overlap.t_screen import color_first_token_ids
from circuit_overlap.patching import (
    LastTokenRunner, capture_last_token_o_proj_inputs, patch_head_last_token, patch_example,
)

_eng = inflect.engine()
WORK = pathlib.Path("/kaggle/working/patching"); WORK.mkdir(parents=True, exist_ok=True)

rows, dropped = lietzow_rows(ds, _eng)
print(len(rows), "objects", dict(dropped), "(Lietzow: 467)")
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(vocab), "colours | sharing a first token (D26):", collisions)

def prompts(r):
    return (VISUAL_PROMPT.format(object=r["name"]),
            PRIOR_PROMPT.format(a_object=_eng.a(r["name"])))

# [VERIFY] on the first object: the cached path equals a full forward pass,
# plain and with one late head patched from the other prompt's run.
r0 = rows[0]
img0 = ds[r0["i"]]["counterfact_image"].convert("RGB")
src0, tgt0 = build_inputs(img0, prompts(r0)[1]), build_inputs(img0, prompts(r0)[0])
source0, target0 = LastTokenRunner(model, src0), LastTokenRunner(model, tgt0)
print("plain    max |dlogit| cached vs full:", target0.check_against_full_forward(atol=1.0))
with capture_last_token_o_proj_inputs(model) as acts0:
    source0.logits()
layer, head = 26, 5
value = acts0[layer][head * head_dim:(head + 1) * head_dim]
with patch_head_last_token(model, layer, head, head_dim, value):
    print("patched  max |dlogit| cached vs full:", target0.check_against_full_forward(atol=1.0))

# Time one patched step to estimate the sweep.
t0 = time.time()
for _ in range(10):
    with patch_head_last_token(model, layer, head, head_dim, value):
        target0.logits()
torch.cuda.synchronize()
step = (time.time() - t0) / 10
print(f"one patched step: {step * 1000:.0f} ms -> ~{73 * 2 * n_layers * n_heads * step / 3600:.1f} h "
      f"of patching for ~73 objects, plus ~{len(rows) * 6 / 60:.0f} min of prompt runs")
del source0, target0, acts0

## 3. The sweep

For every object: run both prompts once (cached), take each run's top-1, and
if the object is correctly conflicting (Visual top-1 = the shown colour, Prior
top-1 = a real colour; same rule as their `load_correct_example_ids`), patch
every head in both directions. One JSON line per object, so a restart resumes.
Source and target prompts must tokenise to the same length, as Lietzow
require; an object where they do not is recorded and skipped.

In [ ]:
def sweep_object(r):
    image = ds[r["i"]]["counterfact_image"].convert("RGB")
    visual_prompt, prior_prompt = prompts(r)
    source_in, target_in = build_inputs(image, prior_prompt), build_inputs(image, visual_prompt)
    if source_in["input_ids"].shape != target_in["input_ids"].shape:
        return {"status": "shape_mismatch"}
    source, target = LastTokenRunner(model, source_in), LastTokenRunner(model, target_in)
    top1_prior, top1_visual = int(source.logits().argmax()), int(target.logits().argmax())
    out = {"name": r["name"], "top1_prior": top1_prior, "top1_visual": top1_visual}
    if not is_correctly_conflicting(r, top1_visual, top1_prior, color_ids, ambiguous):
        return {"status": "not_conflicting", **out}
    record = patch_example(model, source, target, head_dim)
    for key, value in record.items():          # 4 decimals keeps the file ~1.5 MB
        if isinstance(value, list):
            record[key] = [[round(x, 4) for x in row] for row in value]
    return {"status": "patched", **out, **record}

sweep = run_cached(WORK / "patching_records.jsonl", rows, sweep_object, progress_every=10)
from collections import Counter
print(Counter(s["status"] for s in sweep), "(Lietzow: 73 correctly conflicting)")

## 4. Classify the heads and compare with Table 5

`classify_heads` is their PCA +/- 2 sigma rule. Reported against their 9
promoting / 6 suppressing heads: containment (how many of theirs we find),
Jaccard against a permutation null over 576 heads, and the PC1 rank of each of
their heads in our scores.

In [ ]:
import numpy as np
from circuit_overlap.patching import mean_restoration_grids, classify_heads, rank_heads
from circuit_overlap.metrics import containment, jaccard_permutation_test
from circuit_overlap.provenance import result_envelope, write_result
from circuit_overlap.t_screen import native_bf16

patched = [s for s in sweep if s["status"] == "patched"]
p2v, v2p, n_used = mean_restoration_grids(patched)
cls = classify_heads(p2v, v2p, sigma=2.0)
ranks = rank_heads(cls["pc1_scores"])
ours_p, ours_s = set(cls["promoting"]), set(cls["suppressing"])
theirs_p, theirs_s = set(hs.promoting), set(hs.suppressing)
n_total = n_layers * n_heads

print(f"{n_used} objects scored | PC1 direction {cls['pc1_direction']}, "
      f"mean {cls['pc1_mean']:.4f}, std {cls['pc1_std']:.4f}")
print("our promoting  :", sorted(format_head_label(h) for h in ours_p))
print("our suppressing:", sorted(format_head_label(h) for h in ours_s))
print(f"their promoting found by us: {len(theirs_p & ours_p)}/9   "
      f"containment {containment(theirs_p, ours_p):.2f}")
perm = jaccard_permutation_test(ours_p, theirs_p, n_heads=n_total, n_perm=10_000, seed=0)
print(f"Jaccard {perm['jaccard']:.3f}  (chance {perm['null_mean']:.3f}), p = {perm['p_value']:.4f}")
their_ranks = {format_head_label(h): ranks[h] for h in sorted(theirs_p)}
print("PC1 rank of their promoting heads (of 576):", their_ranks,
      " median", float(np.median(list(their_ranks.values()))))
print(f"their suppressing found by us: {len(theirs_s & ours_s)}/6")

envelope = result_envelope(
    "replication_patching",
    {
        "n_objects": len(rows),
        "dropped": dict(dropped),
        "status_counts": dict(Counter(s["status"] for s in sweep)),
        "n_scored": n_used,
        "p2v_grid": p2v.round(6).tolist(),
        "v2p_grid": v2p.round(6).tolist(),
        "pc1_scores": cls["pc1_scores"].round(6).tolist(),
        "pc1_direction": cls["pc1_direction"],
        "pc1_mean": cls["pc1_mean"],
        "pc1_std": cls["pc1_std"],
        "sigma": cls["sigma"],
        "promoting": [list(h) for h in sorted(ours_p)],
        "suppressing": [list(h) for h in sorted(ours_s)],
        "reference_promoting": [list(h) for h in sorted(theirs_p)],
        "reference_suppressing": [list(h) for h in sorted(theirs_s)],
        "records": patched,
    },
    model="Qwen2.5-VL-3B-Instruct",
    checkpoint=config.checkpoint,
    seed=config.seed,
    prompt=f"visual={VISUAL_PROMPT!r} prior={PRIOR_PROMPT!r}",
    n=n_used,
    repo_dir=REPO_DIR,
    quantisation="4-bit nf4, double quant",
    compute_dtype="bfloat16" if native_bf16(torch.cuda.get_device_capability(0)) else "float16",
    gpu=torch.cuda.get_device_name(0),
    image_tokens=n_img,
    method="last-token head patching (o_proj input slice), cached prefix; Lietzow PCA +/-2 sigma",
)
print("\n", write_result(WORK / "replication_patching_qwen25vl3b.json", envelope))
print("Download it and put it in results/; then run scripts/analyze_replication_gate.py.")